# 03b · Information flow: how the situation and the phrase reach the decision

Phase 3a found **where** the decision is written: at layers ≈ 17–20, mostly by **m18** and a few **layer-18 heads**
(a18.h15, h18, h4, h13, …), as a situation–phrase *incongruity* signal. Here we ask **how the information gets there**:

1. **Region patching per layer** (residual stream and attention output): patch the clean activations of one region
   (situation, phrase, question, final token) into the corrupted run. The "staircase" shows at which layer the
   information leaves the changed words and arrives at the final token. Discovery and held-out pairs.
2. **Attention of the last token** for every head, per region, in the four conditions of the 2 × 2 design:
   P (pleasant + positive), S (unpleasant + positive, sarcastic), N (unpleasant + negative), Q (pleasant + negative).
   Which heads read the situation / the phrase, and does that depend on incongruity (S, Q vs P, N)?
3. **What the key heads write along the sarcasm direction** (from 03a), split by the region they read from: does
   a18.h15 bring the *situation* or the *phrase* into the decision?
4. Attention of the top head over the tokens of one example (P, S, N, Q).

**Runtime:** T4; about 20–30 minutes. Needs the results of 03a (direction, writers) on Drive.

## Setup

In [ ]:
# Get (or update) the code from GitHub
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT)
%pip install -q -U accelerate bitsandbytes

In [ ]:
import os, json, time, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src.model import load_model, model_info, DEV_MODEL, MAIN_MODEL
from src.prompts import load_dataset, encode, make_pairs, data_tag, REGIONS
from src.patching import score_pairs, keep_correct, region_patching, mean_and_se
from src.flow import attention_by_region, attention_row
from src.utils import mount_drive, results_dir, cached, set_seed

# ---------------- settings ----------------
MODEL = MAIN_MODEL          # DEV_MODEL (3B) for quick tests
DATA = "v2"
N_RP = 12                   # pairs per experiment and split (discovery / held-out) for region patching
N_ATT_DISC, N_ATT_HELD = 120, 60   # triples for the attention analysis (x 4 conditions)
N_KEY = 8                   # key heads (top writers from 03a)
# ------------------------------------------
TAG_MODEL = MODEL.split("/")[-1]
TAG = TAG_MODEL + data_tag(DATA)
SUB = f"03b_flow/{TAG}"
SUB_3A = f"03a_direction/{TAG}"
mount_drive()
print("results go to:", results_dir(SUB))
set_seed(0)

In [ ]:
t0 = time.time()
model, tok = load_model(MODEL)
INFO = model_info(model)
L, H = INFO["n_layers"], INFO["n_heads"]
print(MODEL, INFO, f"loaded in {(time.time() - t0) / 60:.1f} min")

df = load_dataset(DATA).sort_values(["split", "triple_id", "condition"]).reset_index(drop=True)
df["group"] = [s if s == "discovery" else f"heldout ({t})" for s, t in zip(df.split, df.heldout_type)]

# 03a results: the sarcasm direction and the top writers
S3 = json.load(open(os.path.join(results_dir(SUB_3A), "summary.json")))
L_STAR = S3["L_STAR"]
U = np.load(os.path.join(results_dir(SUB_3A), f"sarcasm_direction_L{L_STAR}.npy"))
ww = pd.read_csv(os.path.join(results_dir(SUB_3A), "who_writes.csv"))
KEY = [c for c in ww.component if ".h" in c][:N_KEY]
KEY_LH = [(int(c[1:].split(".h")[0]), int(c.split(".h")[1])) for c in KEY]
print(f"direction read at layer {L_STAR}; key heads: {KEY}")

## 1 · Region patching per layer (the staircase)

R = 1: patching that region at that layer alone restores the sarcastic answer. Solid: discovery pairs; dashed:
held-out pairs (new phrases / new frame).

In [ ]:
def rp_pairs():
    out = {}
    rng = np.random.default_rng(0)
    for e in ["situation", "phrase"]:
        for split in ["discovery", "heldout"]:
            ps = keep_correct(score_pairs(model, tok, make_pairs(df, tok, e, split=split)))
            ps = [ps[i] for i in rng.permutation(len(ps))][:N_RP]
            out[(e, split)] = ps
    return out

def run_rp():
    res = {}
    for key, ps in rp_pairs().items():
        res[key] = region_patching(model, tok, ps, kinds=("resid", "attn"), progress=False)
        print(f"  {key}: {len(ps)} pairs")
    return res

t0 = time.time()
RP = cached(f"region_patching_n{N_RP}", run_rp, sub=SUB)
print(f"{(time.time() - t0) / 60:.1f} min")

cols = {"situation": "#2E7D32", "phrase": "#C62828", "question": "#6A1B9A", "final": "#1565C0"}
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey=True)
for i, e in enumerate(["situation", "phrase"]):
    for j, kind in enumerate(["resid", "attn"]):
        ax = axes[i, j]
        for split, ls in [("discovery", "-"), ("heldout", "--")]:
            m, se = mean_and_se(RP[(e, split)][kind])
            for r, name in enumerate(REGIONS):
                ax.plot(m[:, r], ls, color=cols[name], lw=2 if split == "discovery" else 1.2,
                        label=f"{name}" if split == "discovery" else None)
        ax.axhline(0, c="gray", lw=0.5); ax.axhline(1, c="gray", lw=0.5, ls=":")
        ax.set_title(f"{e} swap · patch {'residual stream' if kind == 'resid' else 'attention output'}")
        ax.set_xlabel("layer")
    axes[i, 0].set_ylabel("recovery R")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "region_patching.png"), dpi=150); plt.show()

for e in ["situation", "phrase"]:
    m, _ = mean_and_se(RP[(e, "discovery")]["resid"])
    fin = m[:, REGIONS.index("final")]
    chg = m[:, REGIONS.index(e if e == "situation" else "phrase")]
    cross = int(np.argmax(fin > chg)) if (fin > chg).any() else None
    print(f"{e:9s}: the final token carries more than the changed words from layer {cross} "
          f"(R final = {fin[cross]:.2f}, R {e} = {chg[cross]:.2f})" if cross is not None else f"{e}: no crossing")

## 2 · Attention of the last token, per head and region, in P, S, N, Q

Q = pleasant situation + negative phrase (P's sentence with N's phrase), as in 03a.

In [ ]:
rng = np.random.default_rng(1)
tids = list(rng.choice(df[df.group == "discovery"].triple_id.unique(), N_ATT_DISC, replace=False))
tids += list(rng.choice(df[df.split == "heldout"].triple_id.unique(), N_ATT_HELD, replace=False))
sub = df[df.triple_id.isin(tids)].copy()
q_rows = []
for tid, g in sub.groupby("triple_id"):
    r = g.set_index("condition")
    text = r.loc["P", "text"][: -len(r.loc["P", "frame"])] + r.loc["N", "frame"]
    q_rows.append(dict(triple_id=tid, condition="Q", text=text, situation=r.loc["P", "situation"],
                       frame=r.loc["N", "frame"], prompt=r.loc["P", "prompt"].replace(r.loc["P", "text"], text),
                       group=r.loc["P", "group"], split=r.loc["P", "split"]))
AD = pd.concat([sub, pd.DataFrame(q_rows)], ignore_index=True)
ENCS = [encode(tok, r) for r in AD.itertuples()]
WRITE_LAYERS = tuple(range(max(0, L_STAR - 9), L_STAR))
print(f"{len(AD)} prompts ({AD.condition.value_counts().to_dict()}); writes for layers {WRITE_LAYERS}")

t0 = time.time()
ATT, WR = cached(f"attention_n{N_ATT_DISC}_{N_ATT_HELD}", lambda: attention_by_region(model, ENCS, U, WRITE_LAYERS), sub=SUB)
print(f"{(time.time() - t0) / 60:.1f} min | ATT {ATT.shape} WR {WR.shape}")
RN = REGIONS + ["other"]
C = AD.condition.values
mean_c = {c: ATT[C == c].mean(0) for c in "PSNQ"}             # [L, H, 5]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for i, name in enumerate(["situation", "phrase"]):
    r = RN.index(name)
    allc = np.mean([mean_c[c][:, :, r] for c in "PSNQ"], 0)
    incg = (mean_c["S"][:, :, r] + mean_c["Q"][:, :, r]) / 2 - (mean_c["P"][:, :, r] + mean_c["N"][:, :, r]) / 2
    im = axes[i, 0].imshow(allc.T, aspect="auto", origin="lower", cmap="viridis")
    axes[i, 0].set_title(f"attention of the last token to the {name} (mean of P, S, N, Q)")
    plt.colorbar(im, ax=axes[i, 0], shrink=0.8)
    lim = np.abs(incg).max()
    im = axes[i, 1].imshow(incg.T, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[i, 1].set_title(f"to the {name}: incongruent (S, Q) − congruent (P, N)")
    plt.colorbar(im, ax=axes[i, 1], shrink=0.8)
    for ax in axes[i]:
        ax.set_xlabel("layer"); ax.set_ylabel("head")
        for (l, h) in KEY_LH:
            ax.scatter(l, h, s=40, facecolors="none", edgecolors="white" if ax is axes[i, 0] else "black", lw=1.2)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "attention_maps.png"), dpi=150); plt.show()
print("circles = key heads (top writers of the sarcasm direction in 03a)")

rows = []
for (l, h), name in zip(KEY_LH, KEY):
    for c in "PSNQ":
        rows.append(dict(head=name, condition=c, **{rn: mean_c[c][l, h, k] for k, rn in enumerate(RN)}))
KT = pd.DataFrame(rows).set_index(["head", "condition"])
print("attention of the last token (share) for the key heads")
display(KT.round(3))

## 3 · What the key heads write along the sarcasm direction, by the region they read from

For each key head: its projection on the sarcasm direction (read at `L_STAR`), split by source region, per
condition. Below: summed over all heads of each layer, S minus mean(P, N) — which region's information is written
into the decision, layer by layer.

In [ ]:
fig, axes = plt.subplots(2, (len(KEY) + 1) // 2, figsize=(4 * ((len(KEY) + 1) // 2), 7), sharey=True)
for ax, (l, h), name in zip(axes.flat, KEY_LH, KEY):
    if l not in WRITE_LAYERS:
        ax.set_title(f"{name} (not computed)"); continue
    k = WRITE_LAYERS.index(l)
    x = np.arange(len(RN))
    for j, c in enumerate("PSNQ"):
        ax.bar(x + (j - 1.5) * 0.2, WR[C == c][:, k, h].mean(0), 0.2, label=c)
    ax.set_xticks(x); ax.set_xticklabels(RN, rotation=30, fontsize=8); ax.axhline(0, c="gray", lw=0.5)
    ax.set_title(name)
axes.flat[0].set_ylabel("write along the sarcasm direction"); axes.flat[0].legend(fontsize=8)
for ax in list(axes.flat)[len(KEY):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "key_heads_write_by_region.png"), dpi=150); plt.show()

per_layer = WR[C == "S"].sum(2).mean(0) - (WR[C == "P"].sum(2).mean(0) + WR[C == "N"].sum(2).mean(0)) / 2   # [layers, regions]
fig, ax = plt.subplots(figsize=(10, 3.6))
bottom_p, bottom_n = np.zeros(len(WRITE_LAYERS)), np.zeros(len(WRITE_LAYERS))
rc = {"situation": "#2E7D32", "phrase": "#C62828", "question": "#6A1B9A", "final": "#1565C0", "other": "#9E9E9E"}
for k, rn in enumerate(RN):
    v = per_layer[:, k]
    base = np.where(v >= 0, bottom_p, bottom_n)
    ax.bar(WRITE_LAYERS, v, bottom=base, color=rc[rn], label=rn)
    bottom_p += np.clip(v, 0, None); bottom_n += np.clip(v, None, 0)
ax.axhline(0, c="gray", lw=0.6); ax.set_xlabel("layer"); ax.set_ylabel("S − mean(P, N)")
ax.set_title("all attention heads of a layer: write along the sarcasm direction, by source region"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "write_by_region_per_layer.png"), dpi=150); plt.show()

## 4 · The top head on one example

In [ ]:
l0, h0 = KEY_LH[0]
tid = AD[AD.group == "discovery"].triple_id.iloc[0]
fig, axes = plt.subplots(4, 1, figsize=(15, 9))
for ax, c in zip(axes, "PSNQ"):
    i = int(np.flatnonzero((AD.triple_id == tid).values & (C == c))[0])
    e = ENCS[i]
    p = attention_row(model, e.ids, l0, h0)
    start = min(e.regions["situation"]) - 6
    toks = [tok.decode([t]) for t in e.ids[start:]]
    colr = ["#2E7D32" if j in e.regions["situation"] else "#C62828" if j in e.regions["phrase"] else "#9E9E9E"
            for j in range(start, len(e.ids))]
    ax.bar(range(len(toks)), p[start:], color=colr)
    ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=90, fontsize=7)
    ax.set_title(f"{KEY[0]} · {c}: {AD.text.iloc[i]}", fontsize=9)
axes[0].set_ylabel("attention of the last token")
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "top_head_example.png"), dpi=150); plt.show()
print("green = situation, red = phrase")

## Save

In [ ]:
d = results_dir(SUB)
KT.to_csv(os.path.join(d, "key_heads_attention.csv"))
rows = []
for (l, h), name in zip(KEY_LH, KEY):
    if l in WRITE_LAYERS:
        k = WRITE_LAYERS.index(l)
        for c in "PSNQ":
            rows.append(dict(head=name, condition=c, **{rn: float(WR[C == c][:, k, h, j].mean()) for j, rn in enumerate(RN)}))
pd.DataFrame(rows).to_csv(os.path.join(d, "key_heads_write_by_region.csv"), index=False)
pd.DataFrame(per_layer, index=WRITE_LAYERS, columns=RN).to_csv(os.path.join(d, "write_by_region_per_layer.csv"), index_label="layer")
json.dump(dict(model=MODEL, data=DATA, L_STAR=L_STAR, key_heads=KEY), open(os.path.join(d, "summary.json"), "w"), indent=1)
print("saved to", d)